# Synthetic Data, Scaling Laws, and Model Collapse — Tutorial

This notebook accompanies the report **Synthetic Data in the Age of Scaling**. It contains small, transparent simulations of the mechanisms discussed in Julia Kempe's lecture and the associated papers.

These cells are educational reconstructions, not exact replications of the published large-scale experiments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (7, 4)

## 1. Zipf distribution and the long tail

We create a discrete distribution $p(i) \propto i^{-\beta}$. A few ranks have high probability, while many ranks are individually rare.

In [ ]:
beta = 1.5
ranks = np.arange(1, 5001)
p = ranks ** (-beta)
p = p / p.sum()

plt.loglog(ranks, p)
plt.xlabel('rank i')
plt.ylabel('p(i)')
plt.title('Zipf-like long tail')
plt.grid(True, which='both', alpha=0.25)
plt.show()

print('Mass in first 100 ranks:', p[:100].sum())
print('Mass after rank 1000:', p[1000:].sum())

## 2. Hutter infinite-memory missing mass

A learner remembers every observed input exactly and fails only on inputs never seen. The expected error is

$$E_T = \sum_i p_i (1-p_i)^T.$$


In [ ]:
Ts = np.logspace(1, 6, 100).astype(int)
errors = np.array([np.sum(p * (1-p)**T) for T in Ts])

plt.loglog(Ts, errors, label='exact finite support')
plt.loglog(Ts, errors[0]*(Ts/Ts[0])**(-(beta-1)/beta), '--', label='power-law reference')
plt.xlabel('sample size T')
plt.ylabel('missing-mass error')
plt.title('Scaling without abstraction: exact memory baseline')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## 3. Chopped-tail scaling

If synthetic data never contain ranks above $k$, increasing the number of samples cannot remove the missing-support term.

In [ ]:
c = (beta-1)/beta
T = np.logspace(1, 6, 200)
for k in [100, 1000, 10000, 100000]:
    err = T**(-c) + k**(-(beta-1))
    plt.loglog(T, err, label=f'k={k:,}')
plt.xlabel('sample size T')
plt.ylabel('test error')
plt.title('Chopped-tail error floors')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## 4. Mixing scarce real data with synthetic data

Synthetic data can reduce finite-sample error when real data are scarce, but the tail floor remains.

In [ ]:
k = 1000
T_ai = np.logspace(1, 6, 200)
for T_real in [100, 1000, 10000, 100000]:
    err = (T_real + T_ai)**(-c) + k**(-(beta-1))
    plt.loglog(T_ai, err, label=f'T_real={T_real:,}')
plt.xlabel('synthetic samples added')
plt.ylabel('test error')
plt.title('Synthetic data helps most under real-data scarcity')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## 5. Recursive relabeling

The input distribution can remain unchanged while the target parameter drifts because each model labels the next generation.

In [ ]:
x = np.linspace(-3, 3, 150)
w0, b0 = 0.9, 0.15
plt.plot(x, w0*x+b0, linewidth=2.5, label='ground truth')

w, b = w0, b0
rows=[]
for g in range(1, 7):
    w = 0.92*w + rng.normal(0, 0.055)
    b = 0.90*b + rng.normal(0, 0.035)
    plt.plot(x, w*x+b, alpha=0.75, label=f'generation {g}')
    rows.append((g,w,b))

plt.xlabel('x')
plt.ylabel('predicted label')
plt.title('Recursive pseudo-label drift')
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.show()
pd.DataFrame(rows, columns=['generation','slope','intercept'])

## 6. Verification enrichment

Let a generator be correct with probability $q$. A verifier accepts correct candidates with probability $\phi$ and incorrect candidates with probability $\psi$. The accepted-set correctness is calculated with Bayes' rule.

In [ ]:
def accepted_precision(q, phi, psi):
    return phi*q / (phi*q + psi*(1-q))

q = 0.20
for phi, psi in [(0.95,0.05), (0.80,0.10), (0.60,0.20), (0.90,0.30)]:
    print(phi, psi, accepted_precision(q, phi, psi))

## 7. Conceptual MMALS route collapse

This toy simulation illustrates the difference between recursive reinforcement of a dominant host and continued independent anchoring. It is not an empirical MMALS result.

In [ ]:
t = np.arange(60)
h1_recursive = 0.45 + 0.5*(1-np.exp(-t/15))
h2_recursive = 0.25*np.exp(-t/20)
h3_recursive = 0.20*np.exp(-t/18)

h1_anchored = 0.45 + 0.12*(1-np.exp(-t/18))
h2_anchored = 0.22 + 0.03*np.sin(t/8)
h3_anchored = 0.18 + 0.025*np.cos(t/9)

plt.plot(t,h1_recursive,label='H1 recursive-only')
plt.plot(t,h2_recursive,label='H2 recursive-only')
plt.plot(t,h3_recursive,label='H3 recursive-only')
plt.plot(t,h1_anchored,'--',label='H1 anchored')
plt.plot(t,h2_anchored,'--',label='H2 anchored')
plt.plot(t,h3_anchored,'--',label='H3 anchored')
plt.ylim(0,1)
plt.xlabel('learning cycles')
plt.ylabel('route share')
plt.title('Conceptual MMALS functional collapse')
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.show()

## 8. Next experiments

1. Replace the conceptual route shares with actual MMALS traces.
2. Stratify route performance by rare functional regime.
3. Compare recursive-only, fixed-anchor, accumulated-real, and verified-synthetic protocols.
4. Add causal route ablations and route-swap tests.
5. Record lineage, verification, cost, and freshness for each trace.